# Phase 1 Data Cleaning

In [1]:
!pip install pandas numpy

In [2]:
import pandas as pd
import numpy as np

In [5]:
orders = pd.read_csv('orders DA.csv')

In [6]:
print(orders.shape)

(50000, 19)


In [7]:
orders.head()

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
0,ORD000001,CUST02946,22-05-2025,17:10:00,Processing,Pune,Maharashtra,300862,100984.19,16364.64,0,20486.45,100984.19,Cash on Delivery,BlueDart,TRK354572763,NaN,1,App
1,ORD000002,CUST04232,16-10-2023,05:32:00,Delivered,Ludhiana,Punjab,387852,254478.01,41599.80,0,19338.79,254478.01,UPI,DTDC,TRK272838570,19-10-2023,0,Website
2,ORD000003,CUST08078,08-10-2023,12:32:00,Cancelled,Meerut,Uttar Pradesh,294014,32557.06,3562.92,0,696.86,32557.06,Debit Card,Amazon Logistics,TRK338505401,NaN,0,App
3,ORD000004,CUST09612,02-10-2023,08:05:00,Delivered,Nashik,Maharashtra,626047,24836.94,3092.88,0,4029.94,24836.94,UPI,DTDC,TRK289792017,09-10-2023,0,Mobile Web
4,ORD000005,CUST03065,15-06-2024,20:19:00,Delivered,Moradabad,Uttar Pradesh,409210,156689.29,25092.36,0,7805.07,156689.29,UPI,Ecom Express,TRK978240041,22-06-2024,0,App


In [8]:
orders.columns

Index(['order_id', 'customer_id', 'order_date', 'order_time', 'status', 'city',
       'state', 'pincode', 'total_amount', 'gst_amount', 'shipping_charge',
       'discount_amount', 'final_amount', 'payment_method', 'shipping_partner',
       'tracking_id', 'delivered_date', 'is_cod', 'channel'],
      dtype='object')

In [9]:
orders.dtypes

order_id             object
customer_id          object
order_date           object
order_time           object
status               object
city                 object
state                object
pincode               int64
total_amount        float64
gst_amount          float64
shipping_charge       int64
discount_amount     float64
final_amount        float64
payment_method       object
shipping_partner     object
tracking_id          object
delivered_date       object
is_cod                int64
channel              object
dtype: object

In [11]:
orders['order_date'] = pd.to_datetime(orders['order_date'], format='%d-%m-%Y')
orders['order_date'].dtype

dtype('<M8[ns]')

In [12]:
files = {
    'orders':      'orders DA.csv',
    'order_items': 'order_items DA.csv',
    'customers':   'customers DA.csv',
    'products':    'products DA.csv',
    'payments':    'payments DA.csv',
    'returns':     'returns DA.csv',
    'inventory':   'inventory DA.csv',
    'suppliers':   'suppliers DA.csv',
}

dfs = {name: pd.read_csv(path) for name, path in files.items()}

In [13]:
orders      = dfs['orders']
order_items = dfs['order_items']
customers   = dfs['customers']
products    = dfs['products']
payments    = dfs['payments']
returns     = dfs['returns']
inventory   = dfs['inventory']
suppliers   = dfs['suppliers']

In [14]:
for name, df in dfs.items():
    print(name, df.shape)

orders (50000, 19)
order_items (100000, 11)
customers (10000, 17)
products (1000, 17)
payments (50000, 13)
returns (10000, 10)
inventory (1000, 11)
suppliers (200, 14)


In [15]:
nulls = orders.isnull().sum()
print(nulls)

order_id                0
customer_id             0
order_date              0
order_time              0
status                  0
city                    0
state                   0
pincode                 0
total_amount            0
gst_amount              0
shipping_charge         0
discount_amount         0
final_amount            0
payment_method          0
shipping_partner        0
tracking_id             0
delivered_date      17501
is_cod                  0
channel                 0
dtype: int64


In [16]:
for name, df in dfs.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"{name}: no missing values")
    else:
        print(f"\n{name}:")
        print(nulls)


orders:
delivered_date    17501
dtype: int64
order_items: no missing values
customers: no missing values
products: no missing values

payments:
refund_date    50000
dtype: int64
returns: no missing values
inventory: no missing values
suppliers: no missing values


In [18]:
payments.columns

Index(['payment_id', 'order_id', 'customer_id', 'payment_date', 'payment_time',
       'payment_method', 'amount', 'status', 'transaction_id', 'bank_name',
       'gateway', 'refund_amount', 'refund_date'],
      dtype='object')

In [19]:
payments['refund_amount'].describe()


count    50000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: refund_amount, dtype: float64

In [20]:
print("orders duplicate order_id:", orders['order_id'].duplicated().sum())
print("customers duplicate customer_id:", customers['customer_id'].duplicated().sum())
print("products duplicate product_id:", products['product_id'].duplicated().sum())

orders duplicate order_id: 0
customers duplicate customer_id: 0
products duplicate product_id: 0


In [21]:
print("orders fully duplicated rows:", orders.duplicated().sum())

orders fully duplicated rows: 0


In [22]:
outliers = orders[orders['final_amount'] > 500000]
print("Number of outlier orders:", len(outliers))
print("Highest final_amount:", orders['final_amount'].max())

Number of outlier orders: 312
Highest final_amount: 1133382.48


In [23]:
order_items.columns

Index(['item_id', 'order_id', 'product_id', 'product_name', 'category',
       'quantity', 'unit_price', 'gst_rate', 'gst_amount', 'discount_amount',
       'total_price'],
      dtype='object')

In [24]:
bad_links = ~order_items['order_id'].isin(orders['order_id'])
print("order_items rows with no matching order:", bad_links.sum())

order_items rows with no matching order: 0


In [27]:
print("product_id rows with no matching product:", (~order_items['product_id'].isin(products['product_id'])).sum())
print("customer_id rows with no matching product:", (~orders['customer_id'].isin(customers['customer_id'])).sum())
print("order_id rows with no matching product:", (~payments['order_id'].isin(orders['order_id'])).sum())
print("order_id rows with no matching product:", (~returns['order_id'].isin(orders['order_id'])).sum())

product_id rows with no matching product: 0
customer_id rows with no matching product: 0
order_id rows with no matching product: 0
order_id rows with no matching product: 0


In [28]:
orders_without_items = ~orders['order_id'].isin(order_items['order_id'])
print("orders with zero order_items:", orders_without_items.sum())

orders with zero order_items: 10071


In [29]:
has_delivered = orders['delivered_date'].notna()
invalid_dates = orders[has_delivered & (orders['delivered_date'] < orders['order_date'])]
print("Orders with a delivered_date:", has_delivered.sum())
print("Invalid rows (delivered before ordered):", len(invalid_dates))

Orders with a delivered_date: 32499
Invalid rows (delivered before ordered): 4883


In [30]:
orders['delivered_date'].dtype


dtype('O')

In [31]:
orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], format='%d-%m-%Y', errors='coerce')
orders['delivered_date'].dtype

dtype('<M8[ns]')

In [32]:
has_delivered = orders['delivered_date'].notna()
invalid_dates = orders[has_delivered & (orders['delivered_date'] < orders['order_date'])]
print("Orders with a delivered_date:", has_delivered.sum())
print("Invalid rows (delivered before ordered):", len(invalid_dates))

Orders with a delivered_date: 32499
Invalid rows (delivered before ordered): 5790


In [33]:
orders['order_date'].dtype


dtype('O')

In [34]:
orders['order_date'] = pd.to_datetime(orders['order_date'], format='%d-%m-%Y')
orders['order_date'].dtype

dtype('<M8[ns]')

In [35]:
has_delivered = orders['delivered_date'].notna()
invalid_dates = orders[has_delivered & (orders['delivered_date'] < orders['order_date'])]
print("Orders with a delivered_date:", has_delivered.sum())
print("Invalid rows (delivered before ordered):", len(invalid_dates))

Orders with a delivered_date: 32499
Invalid rows (delivered before ordered): 0


In [36]:
print(payments['status'].value_counts())

failure_rate = (payments['status'] == 'Failed').sum() / len(payments) * 100
print("Failure rate:", failure_rate)

status
Success    48252
Failed      1748
Name: count, dtype: int64
Failure rate: 3.496


In [37]:
print(inventory['status'].value_counts())

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64


## Data Quality Report — IndiaKart Phase 1

The IndiaKart dataset (8 tables, ~216,200 records, June 2023 to June 2025) is in good shape overall.
I ran the full set of checks the PRD asked for — nulls, duplicates, data types, outliers, and whether
the tables actually link up to each other correctly — and the short version is: nothing here should
stop us from moving into EDA and KPI work. A couple of things are worth flagging though, so leaving
them here rather than letting them surprise someone later.

### Missing Values

Missing values were limited to just two columns. `orders.delivered_date` is empty for 17,501 rows (35%),
but that's not a data problem — it just means those orders haven't been delivered yet (they're sitting
in Cancelled, Shipped, Processing, or Returned status). `payments.refund_date` is empty across all 50,000
rows, which looked alarming at first, until I checked `refund_amount` and found it's 0 for every single
payment too. So this column just isn't used — refunds are actually tracked over in `returns.csv` instead.
Every other column in every other table was fully populated.

### Duplicates

No duplicates anywhere — checked primary keys across all the ID columns (`order_id`, `customer_id`,
`product_id`, `item_id`, `payment_id`, `return_id`) and also checked for fully duplicated rows in `orders`.
Came back clean.

### Data Types

Data types needed some work. Dates load as plain text by default, so `order_date` and `delivered_date`
both had to be converted properly. This actually caught a real bug along the way: before I converted
both columns, comparing `delivered_date` to `order_date` showed 5,790 orders where delivery apparently
happened before the order was placed, which obviously isn't possible. Turned out this was just Python
comparing the dates as text instead of as actual dates. Once both columns were converted to proper
datetimes, that number dropped to 0 — so there's no real issue here, just a reminder that date columns
need to be typed correctly before you trust any comparison on them.

### Referential Integrity

The tables link up to each other correctly — checked `order_items` against `orders`, `order_items` against
`products`, `orders` against `customers`, `payments` against `orders`, and `returns` against `orders`, and
every single foreign key resolved with zero broken links. The one thing worth calling out: **10,071 orders
(about 20%)** don't have any matching rows in `order_items` at all. That doesn't break anything at the
order level — GMV, AOV, cancellation rate all come straight from `orders.final_amount` and are fine —
but it does mean any category or product-level breakdown later on is really only based on ~80% of
orders, not all of them. Worth keeping in mind for Phase 2/3 so nobody's surprised by the coverage gap.

### Outliers

312 orders (0.62%) came in above ₹5 lakh, the highest being ₹11,33,382. I'm not treating these as
errors — high-value electronics orders are believable — but they're flagged here in case someone wants
to spot-check a few before they go into revenue totals.

### Business Risk Flags

Two numbers are worth flagging as actual business risks rather than data issues.

- **Payment failure rate: 3.5%** (1,748 of 50,000 transactions) — above the <2% target set in the brief,
  meaning real revenue is slipping through the cracks.
- **Inventory fill rate: 92.2%** — just under the >95% target, though it's mostly driven by 76 products
  sitting in Low Stock rather than genuine stockouts (only 2 products are fully Out of Stock).

### Final Call on Phase 1

The data's ready to move forward with. Just carrying the `order_items` coverage gap as a caveat into the
next phase, and flagging the payment failure rate and inventory fill rate as two things worth putting in
front of management later.